In [2]:
import wandb
import pandas as pd


Retrieve data from Wandb

In [6]:

# Replace these with your actual values
entity = "budisicm-virginia-commonwealth-university"           # e.g., your username or team name
project = "huggingface"    # e.g., "ppo-cartpole"

run_names = {
    "original":"t4sdhzwc",
    "Fix-soft":"39g5f5hv",
    "Fix-both":"hvqfw22r"
}

api = wandb.Api()
run_history = {}

for run_name, run_id in run_names.items():
    run = api.run(f"{entity}/{project}/{run_id}")

    # Fetch full history of the run (use `keys` to limit what you download)
    run_history[run_name] = run.history(samples=176)  # You can increase samples if needed

reward = pd.concat({key: df['train/reward'] for key, df in run_history.items()}, axis=1)
xml_count = pd.concat({key: df['train/rewards/xmlcount_reward_func'] for key, df in run_history.items()}, axis=1)

Import graphics libraries

In [4]:
import plotly.graph_objects as go
import plotly.colors as pc


In [10]:
def visualize(dataframe, title,tick=0.25):
    fig = go.Figure()
    window_size = 20
    colors = pc.qualitative.Safe  # Or Set2, Set3, etc.
    keys = list(dataframe.columns)

    fig = go.Figure()

    # Plot raw and average with same color
    for i, col in enumerate(keys):

        color = colors[i % len(colors)]
        fig.add_trace(go.Scatter(
            y=dataframe[col],
            mode='lines',
            name=f'{col} (raw)',
            opacity=0.5,
            line=dict(width=1, color=color)
        ))
        
        fig.add_trace(go.Scatter(
            y=dataframe[col].rolling(window=window_size, min_periods=1).mean(),
            mode='lines',
            name=f'{col} (avg)',
            opacity=1.0,
            line=dict(width=2, color=color)
        ))

    # Update layout
    fig.update_layout(
        title=f'Original Values and Running Averages (w={window_size})',
        xaxis_title='Step',
        yaxis_title=title,
        legend=dict(
            orientation='h',  # horizontal legend
            x=0,
            y=1.1,
            xanchor='left',
            yanchor='bottom'
        ),
        template='plotly_white',
        yaxis=dict(
            tick0=0,
            dtick=tick,  # sets the interval between ticks
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',        
            mirror='all',
            ticks="outside",
            showline=True,
            side='right'
        )
    )

    fig.show()


In [11]:
visualize(reward, 'train/reward',tick=0.25)


In [12]:
visualize(xml_count, 'train/rewards/xmlcount_reward_func',tick=0.125)
